# 6. Callback subset

Reports with a narrative and a real callback (callbacks starting with *“Unable to contact…”* are not counted): **4,698 reports**.

### Run the earlier steps
Loads the variables this notebook needs (`df`, `df_norm`, ...) by running `05_data_quality_checks.ipynb` (which runs the steps before it). Output is hidden.

In [1]:
%%capture
%run -i "05_data_quality_checks.ipynb"

## 6.1 Callback diagnostic 1: report counts vs. target counts

In [2]:
# Callback diagnostic 1: report counts versus target counts
# Run after loading the full CSV into df. Nothing in df is changed.

if not isinstance(globals().get("df"), pd.DataFrame) or df.empty:
    raise ValueError("Load the full C3RS CSV into df first.")

cb_headers = [" ".join(str(c).split()) for c in df.columns]
if len(cb_headers) != len(set(cb_headers)):
    raise ValueError("Column names collide after whitespace normalization.")

cb_lookup = dict(zip(cb_headers, df.columns))
cb_narrative_cols = [
    c for c in cb_headers
    if re.fullmatch(r"Person Narrative / Narrative \d+", c)
]
cb_callback_cols = [
    c for c in cb_headers
    if re.fullmatch(r"Person Callback / Callback \d+", c)
]
cb_anomaly_cols = [c for c in cb_headers if c.startswith("Events Anomaly /")]
cb_target = "Events Anomaly / Safety Concern"

if not cb_narrative_cols or not cb_callback_cols or cb_target not in cb_lookup:
    raise KeyError("Expected narrative, callback, or Safety Concern fields are missing.")

cb_keep = cb_narrative_cols + cb_callback_cols + cb_anomaly_cols
cb_review = df[[cb_lookup[c] for c in cb_keep]].copy().reset_index(drop=True)
cb_review.columns = cb_keep

cb_narratives = cb_review[cb_narrative_cols].astype("string").fillna("").apply(
    lambda s: s.str.strip()
)
cb_callbacks = cb_review[cb_callback_cols].astype("string").fillna("").apply(
    lambda s: s.str.strip()
)

# Recognize observed supplemental-person placeholders, not short narratives.
cb_placeholder = cb_narratives.apply(
    lambda s: s.str.fullmatch(
        r"\[?(?:(?:second|third) person|person \d+)(?: reference)? "
        r"added to supplement the records?\.?\]?\.?",
        case=False, na=False
    )
)
cb_has_narrative = (cb_narratives.ne("") & ~cb_placeholder).any(axis=1)

# This reproduces the earlier "Unable to contact..." screening rule.
# It is a candidate-text screen, not proof that other callbacks are informative.
# Original contact notices and all other source text remain in df.
cb_contact_notice = cb_callbacks.apply(
    lambda s: s.str.lower().str.startswith("unable to contact")
)
cb_has_callback_text = (
    cb_callbacks.ne("") & ~cb_contact_notice
).any(axis=1)
cb_pair_mask = cb_has_narrative & cb_has_callback_text

cb_true_values = ["true", "1", "1.0", "yes"]
cb_false_values = ["false", "0", "0.0", "no"]
cb_target_values = (
    cb_review[cb_target].astype("string").fillna("").str.strip().str.lower()
)
if not cb_target_values.isin(cb_true_values + cb_false_values + [""]).all():
    raise ValueError("Unrecognized target values: review them before coding.")
cb_target_selected = cb_target_values.isin(cb_true_values)

cb_cohorts = {
    "Full export": pd.Series(True, index=cb_review.index),
    "Narrative + candidate callback": cb_pair_mask,
    "Other reports retained in master": ~cb_pair_mask,
}
cb_cohort_rows = []
for name, mask in cb_cohorts.items():
    total = int(mask.sum())
    n_selected = int((mask & cb_target_selected).sum())
    cb_cohort_rows.append({
        "Report group": name,
        "Total reports": total,
        "Safety Concern selected": n_selected,
        "Safety Concern not selected": total - n_selected,
        "Selected (%)": 100 * n_selected / total if total else float("nan"),
    })

cb_cohort_table = pd.DataFrame(cb_cohort_rows).set_index("Report group")
display(cb_cohort_table.round(2))
print("Not selected means this code is absent, not that the report is safe.")
print("The callback group is a SUBSET of the full export, not extra positive labels.")
print("No records were removed; no training or new split was performed.")


,Total reports,Safety Concern selected,Safety Concern not selected,Selected (%)
Report group,,,,
Full export,10261,411,9850,4.01
Narrative + candidate callback,4698,235,4463,5.00
Other reports retained in master,5563,176,5387,3.16


Not selected means this code is absent, not that the report is safe.
The callback group is a SUBSET of the full export, not extra positive labels.
No records were removed; no training or new split was performed.


## 6.2 Callback diagnostic 2: every recorded event category

In [3]:
# Callback diagnostic 2: inspect every original anomaly category
# Run after Callback diagnostic 1. This does not select the study's target.
cb_values = cb_review[cb_anomaly_cols].astype("string").fillna("").apply(
    lambda s: s.str.strip().str.lower()
)
cb_recognized = cb_true_values + cb_false_values + [""]
cb_open_columns = [
    c for c in cb_anomaly_cols if not cb_values[c].isin(cb_recognized).all()
]
cb_code_columns = [c for c in cb_anomaly_cols if c not in cb_open_columns]
cb_selected_codes = cb_values[cb_code_columns].isin(cb_true_values)

cb_category_table = pd.DataFrame({
    "Full selected": cb_selected_codes.sum(),
    "Callback-group selected": cb_selected_codes.loc[cb_pair_mask].sum(),
})
cb_category_table["Callback-group not selected"] = (
    int(cb_pair_mask.sum()) - cb_category_table["Callback-group selected"]
)
cb_category_table["Full selected (%)"] = (
    100 * cb_category_table["Full selected"] / len(cb_review)
)
cb_category_table["Callback-group selected (%)"] = (
    100 * cb_category_table["Callback-group selected"] / int(cb_pair_mask.sum())
    if cb_pair_mask.any() else float("nan")
)
cb_category_table.index = cb_category_table.index.str.replace(
    "Events Anomaly / ", "", regex=False
)
cb_category_table.index.name = "Original event/anomaly category"
cb_category_table = cb_category_table.sort_values(
    "Callback-group selected", ascending=False
)

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(cb_category_table.round(2))

# Open-text Other fields are not silently interpreted as Boolean outcomes.
if cb_open_columns:
    print("\nOpen-text or unresolved fields excluded from the Boolean count table:")
    for column in cb_open_columns:
        print(
            f"{column}: "
            f"{int(cb_values[column].ne('').sum()):,} nonblank entries"
        )

print("\nA report can have several codes; do not add category counts as unique reports.")
print("Large counts alone do not establish a useful or well-defined target.")


,Full selected,Callback-group selected,Callback-group not selected,Full selected (%),Callback-group selected (%)
Original event/anomaly category,,,,,
Procedural Deviation - Operations Policy,10103,4597,101,98.46,97.85
Operating Cab / Car Event - Station Operations,2541,1227,3471,24.76,26.12
Speed Deviation - Maximum Authorized Speed,2042,873,3825,19.90,18.58
Braking Event - Emergency / Penalty Brake Application,1848,850,3848,18.01,18.09
Mechanical - Less Severe,1657,757,3941,16.15,16.11
Signal Events - On Board Control System,1430,644,4054,13.94,13.71
Speed Deviation - At or Above 10MPH Deviation,1419,612,4086,13.83,13.03
Speed Deviation - Temporary Speed Restriction,1373,565,4133,13.38,12.03
On Track Protection Deviation - Visual Or Audible Protection,878,381,4317,8.56,8.11



Open-text or unresolved fields excluded from the Boolean count table:
Events Anomaly / Braking Event - Other: 35 nonblank entries
Events Anomaly / Encounter - Other: 6 nonblank entries
Events Anomaly / Operating Cab / Car Event - Other: 24 nonblank entries
Events Anomaly / Procedural Deviation - Other: 16 nonblank entries
Events Anomaly / Other: 55 nonblank entries

A report can have several codes; do not add category counts as unique reports.
Large counts alone do not establish a useful or well-defined target.
